# MVP2 FDTD -> HDMI (4-LANE quad)
Same EM-wave-as-3D-terrain demo, but the FDTD solver is **4-lane parallel** (Taha's quad architecture): one iteration finishes in **2048 cycles instead of 8192** (4x). Single source this build.

AXI map: renderer camera `0x40000000` (optional) | `gpio_ctrl 0x41200000` | `gpio_status 0x41210000`.
`gpio_ctrl` CH2 = `{clear_req[22], mag_mode_hi[21], height_ctl[20:16], free_run[15], sample_req[14], mag_mode_lo[13], solver_enable[12], source_addr[11:0]}`.


In [ ]:
from pynq import Overlay, MMIO
import time
ol = Overlay('fdtd_hdmi.bit')
print('overlay loaded'); print(list(ol.ip_dict.keys()))


In [ ]:
CTRL   = MMIO(0x41200000, 0x10000)
STATUS = MMIO(0x41210000, 0x10000)
GPIO_CH1, GPIO_CH2 = 0x0, 0x8
GRID = 64
def cell(x, y): return y*GRID + x
def q313(v):    return int(round(v*8192)) & 0xFFFF

def set_ctrl(phase_step, amplitude, source_addr, solver_enable, mag_mode,
             sample_req, free_run, height_ctl=2):
    """mag_mode: 0=|E|, 1=|S|, 2=raw signed Ey (smooth waves). height_ctl signed -16..15."""
    ch1 = (q313(amplitude) << 16) | q313(phase_step)
    ch2 = (((mag_mode >> 1) & 1) << 21) | ((height_ctl & 0x1F) << 16) | ((free_run & 1) << 15) \
        | ((sample_req & 1) << 14) | ((mag_mode & 1) << 13) | ((solver_enable & 1) << 12) \
        | (source_addr & 0xFFF)
    CTRL.write(GPIO_CH1, ch1); CTRL.write(GPIO_CH2, ch2)

def set_height(h):
    ch2 = (CTRL.read(GPIO_CH2) & ~(0x1F<<16)) | ((h & 0x1F)<<16); CTRL.write(GPIO_CH2, ch2)

def clear_fields():
    """Reset the simulation: zero Ey/Ex/Bz (all 4 lanes). Edge-triggered bit 22."""
    v = CTRL.read(GPIO_CH2)
    CTRL.write(GPIO_CH2, v | (1<<22)); time.sleep(0.005); CTRL.write(GPIO_CH2, v & ~(1<<22))
    print("fields cleared")

def read_status():
    chk = STATUS.read(GPIO_CH1); s = STATUS.read(GPIO_CH2)
    return dict(checksum=chk, solver_done=(s>>0)&1, source_valid=(s>>1)&1, mag_done=(s>>2)&1,
                mag_busy=(s>>3)&1, source_latched=(s>>4)&1, pp_read_sel=(s>>5)&1,
                pp_frame_ready=(s>>6)&1, bridge_busy=(s>>7)&1, source_q313=(s>>16)&0xFFFF)
print("helpers ready")


## 1. Start free-run (4-lane solver)


In [ ]:
set_ctrl(phase_step=0.01, amplitude=0.3, source_addr=cell(32,32),
         solver_enable=1, mag_mode=2, sample_req=1, free_run=1, height_ctl=2)
print("4-lane FDTD running. Tune relief with set_height(n); reset with clear_fields().")


## 2. Confirm alive
Checksum must keep changing and ping-pong must swap. The solver now produces frames ~4x faster than the single-lane build.


In [ ]:
seen=set(); flips=0; prev=None
for _ in range(20):
    st=read_status(); seen.add(st['checksum'])
    if prev is not None and st['pp_read_sel']!=prev: flips+=1
    prev=st['pp_read_sel']; time.sleep(0.05)
print("unique checksums:", len(seen), " read_sel flips:", flips)
assert len(seen)>1, "checksum frozen"
print("PASS: 4-lane solver live.")


## 3. Reset / clean the renderer


In [ ]:
clear_fields()   # zero the field, wave restarts from the source


## 4. HDMI
Expect the **same** 3D wave terrain as the single-lane build (try `mag_mode=2` signed-Ey). The win is internal: the solver advances 4x faster per iteration. Tuning: `set_height(1..4)`, `phase_step` for wavelength, `mag_mode` 0/1/2 for view.
